# NB15 — UNIFIED COMPUTATIONAL BENCHMARK (TABLE 8B) | NIR-HUEVOS 2026

**Reviewer-driven re-measurement of Table 8B in ONE consistent environment.**

## Why this notebook exists
Reviewer 1 objected that the original Table 8B compared the latency of compiled C++ backends
(scikit-learn PLSR/SVR) against a default TensorFlow CPU graph, and that presenting this as an
accuracy-latency trade-off implies the recurrent models are inherently inefficient rather than
simply unoptimized for that environment.

Separately, CNN1D was added after the primary workflow was frozen, so it has **no** latency,
throughput, or model-size measurement at all — it appears in Tables 3, 5A and 8A but not 8B.

NB15 resolves both by measuring **all seven models in a single session on a single machine**,
under an identical timing protocol, and emitting a complete replacement Table 8B.

## What it measures
For each of SVR, PLSR, ANN, SimpleRNN, LSTM, BiLSTM and CNN1D:
- complexity measure and value (support vectors / latent components / trainable parameters)
- serialized model size on disk (MB), measured the same way for every model
- single-spectrum CPU inference latency (ms/spectrum, batch size 1)
- throughput (spectra/s)

## Protocol controls
- **CPU only.** GPU is explicitly disabled so that all seven models are timed on the same device.
  Latency, not training, is the quantity of interest.
- Models are fitted on outer fold 1's 24 training eggs using the **frozen NB03/NB04/NB11
  preprocessing and epoch selections** — no retuning, no new model selection.
- Timing uses a warm-up phase (discarded) followed by `N_TIMING_REPEATS` timed single-spectrum
  predictions; the reported value is the **median**, which is robust to scheduler noise.
- Every model is timed in the same loop, in the same process, on the same input.
- Outer-test eggs are used only as timing inputs, never for fitting or selection.

## Important
This notebook **only** re-measures the computational descriptors of Table 8B. It does **not**
touch any predictive result: MAE, RMSE, R², the statistical tests and every frozen OOF
prediction stay exactly as they are. Table 8A is unaffected.

## Revision note (v6)
Two consecutive runs of this notebook landed on different Colab-assigned CPUs (an AMD EPYC 7B12,
then an Intel Xeon Broadwell), and absolute latencies differed by roughly 2x between them, with
ANN and SVR landing close enough on one run for their Figure 6 labels to overlap. `ANN` now has
the same small label offset as `PLSR` and `SimpleRNN` so this does not recur. This also means:
Colab's "CPU-only" setting fixes the *device type*, not the *specific CPU model* - the hardware
actually used for a given run is recorded in `hardware_info.txt` in the result package and should
be quoted alongside the numbers in the manuscript. The comparison within one run remains fair
(all seven models on the same machine, same process, same protocol); it is only the exact
milliseconds that should be expected to shift somewhat between separate runs.

## Revision note (v5)
Figure 6 of the manuscript is now generated by this notebook (cell below, before the packaging
step) rather than drawn separately, so the published figure is reproducible from the repository.
Both axes are read from frozen CSV outputs and nothing is hard-coded: latency comes from this
notebook's own benchmark, and pooled seed-mean MAE comes from the frozen seven-model performance
table. The figure is written to the results directory as 600-dpi PNG and TIFF and is included in
the result package.

## Revision note (v4) - READ THIS FIRST

v3 contained a bug. Its "direct" path was documented as tf.function-wrapped but the code used a
plain Python lambda, so it never compiled to a graph. The recurrent models therefore ran in pure
eager mode, dispatching all 331 sequential timesteps individually from Python. That is why v3
reported BiLSTM at 2397 ms on the "direct" path but only 108 ms through `predict()`.

Neither v3 column is a valid architectural comparison:
- `predict()` compiles to a graph (good for RNNs) but adds ~85-108 ms of per-call dispatch
  overhead that swamps the real differences.
- eager direct call removes that overhead (ANN drops to 5.4 ms) but penalises RNNs with
  per-timestep eager dispatch.

v4 measures **three paths** so the sources of difference are separable and auditable:
1. **`predict()`** - the high-level API.
2. **eager** - `model(x, training=False)` called directly, no graph.
3. **graph** - the same call wrapped in `tf.function` with a fixed `input_signature`, traced
   once during warm-up. **This is the column to use in Table 8B**: compiled execution without
   the `predict()` wrapper, which is how inference is actually deployed.

Input tensors are created before the timing loop so tensor construction is not measured.
For scikit-learn all three paths are the same call and the columns coincide by construction.

## Revision note (v3, superseded)
The v2 run exposed a measurement artifact: every Keras model landed at 108-134 ms/spectrum,
including a 23,361-parameter ANN. That floor is `model.predict()` per-call dispatch overhead,
not architecture compute. Comparing `sklearn.predict` (thin wrapper) against `keras.predict`
(heavy per-call machinery) reproduces exactly the criticism this notebook was written to answer.

v3 therefore times **two inference paths for every model** and reports both:
- **`predict()` path** - the high-level API, what the v2 run measured.
- **direct-call path** - `model(x, training=False)` for Keras (the standard low-overhead
  inference call) and the plain `.predict()` for scikit-learn, which has no equivalent
  high-level wrapper.

The gap between the two columns *is* the framework overhead, made explicit rather than hidden.
The direct-call column is the one to use for architectural comparison; the `predict()` column
shows how much of an apparent latency difference is implementation rather than model.

## Revision note (v2)
Frozen selection files are now located by searching the project tree, because NB11's outputs live
under the reviewer-round package rather than beside NB03/NB04. The PLSR/SVR configuration is read
using the real NB03 column names and there is **no fallback default**: fold 1 uses epsilon = 1.0,
not 0.05, and a wrong epsilon would silently change the support-vector count and the measured SVR
latency. Schema assumptions are asserted up front so the run fails loudly instead of guessing.

## Runtime
CPU runtime is sufficient and is in fact **required**. Expect roughly 15-25 minutes, dominated
by fitting SimpleRNN/LSTM/BiLSTM once each on CPU.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Force CPU-only BEFORE importing TensorFlow, so all seven models are timed on one device
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_DETERMINISTIC_OPS'] = '1'

from pathlib import Path
from datetime import datetime, timezone
import gc, hashlib, json, platform, random, shutil, subprocess, sys, time, warnings

import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings('ignore')

assert len(tf.config.list_physical_devices('GPU')) == 0, (
    'GPU still visible. Restart the runtime and run this cell FIRST, before any TF import.'
)
print('TensorFlow:', tf.__version__, '| devices:', [d.device_type for d in tf.config.list_physical_devices()])

PROJECT_ROOT = Path('/content/drive/MyDrive/NIR_HUEVOS_PAPER_REBUILD_2026')
RAW_DIR      = PROJECT_ROOT / '01_DATA_RAW'
SPLIT_DIR    = PROJECT_ROOT / '03_SPLITS_FROZEN'
RESULT_DIR   = PROJECT_ROOT / '05_RESULTS' / 'NB15_UNIFIED_COMPUTATIONAL_BENCHMARK'
ZIP_DIR      = PROJECT_ROOT / '05_RESULTS' / 'ZIP_PACKAGES'
NOTEBOOKS_DIR= PROJECT_ROOT / '04_NOTEBOOKS'
for p in [RESULT_DIR, ZIP_DIR, NOTEBOOKS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DATA_FILE  = RAW_DIR / 'dataset_egg_storage_RAW.csv'
OUTER_FILE = SPLIT_DIR / 'outer_group_assignment_seed2026.csv'
SPLIT_MANIFEST_FILE = SPLIT_DIR / 'split_manifest.json'
# Frozen selection files are located by search: their folders differ between the primary
# workflow and the reviewer-round package, so hard-coded paths are unreliable.
def find_one(filename, root=PROJECT_ROOT):
    hits = sorted(root.rglob(filename))
    assert hits, f'Not found anywhere under {root}: {filename}'
    if len(hits) > 1:
        print(f'  note: {len(hits)} copies of {filename}; using the first')
        for h in hits:
            print('    -', h.relative_to(root))
    return hits[0]

NB03_SELECTED_FILE = find_one('NB03_selected_configurations.csv')
NB04_SELECTED_FILE = find_one('NB04_selected_configurations.csv')
NB11_SELECTED_FILE = find_one('NB11_selected_configurations.csv')
print('NB03 selections:', NB03_SELECTED_FILE.relative_to(PROJECT_ROOT))
print('NB04 selections:', NB04_SELECTED_FILE.relative_to(PROJECT_ROOT))
print('NB11 selections:', NB11_SELECTED_FILE.relative_to(PROJECT_ROOT))

EXPECTED_DATASET_SHA256 = 'cd5021c555ae6b57f892549c574599cef75edf87f58b3f7f4d246ade9327d15e'
EXPECTED_SPLIT_MANIFEST_SHA256 = 'fbeb8fa19d522cd91bee875bf5731cda264475da27bc7e93c25ca0d6f0f33717'

NOTEBOOK_FILENAME = 'NB15_UNIFIED_COMPUTATIONAL_BENCHMARK.ipynb'
RUN_REVISION = 'NB15_v6_three_path_cpu_latency_plus_figure6'

TIMING_FOLD        = 1      # outer fold used to fit the timed models
N_TIMING_WARMUP    = 30     # discarded predictions
N_TIMING_REPEATS   = 200    # timed single-spectrum predictions per model
SG_WINDOW, SG_POLYORDER = 11, 2
BATCH_SIZE, LEARNING_RATE, DROPOUT = 32, 1e-3, 0.20
RECURRENT_UNITS, DENSE_AFTER_RECURRENT, ANN_HIDDEN = 64, 32, [64, 32]

MODELS = ['SVR', 'PLSR', 'ANN', 'SimpleRNN', 'LSTM', 'BiLSTM', 'CNN1D']
print('CPU-only mode confirmed.')


In [ ]:
# Integrity gate — same frozen inputs as every other notebook in this project
def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

for p in [DATA_FILE, OUTER_FILE, SPLIT_MANIFEST_FILE,
          NB03_SELECTED_FILE, NB04_SELECTED_FILE, NB11_SELECTED_FILE]:
    assert p.exists(), f'Missing required input: {p}'

dataset_sha = sha256_file(DATA_FILE)
split_sha   = sha256_file(SPLIT_MANIFEST_FILE)
assert dataset_sha == EXPECTED_DATASET_SHA256, 'Dataset hash changed.'
assert split_sha == EXPECTED_SPLIT_MANIFEST_SHA256, 'Frozen split manifest hash changed.'

manifest = json.loads(SPLIT_MANIFEST_FILE.read_text(encoding='utf-8'))
for fname, expected in manifest['files'].items():
    fp = SPLIT_DIR / fname
    assert fp.exists() and sha256_file(fp) == expected, f'Frozen split changed: {fname}'

sel_nb03 = pd.read_csv(NB03_SELECTED_FILE)
sel_nb04 = pd.read_csv(NB04_SELECTED_FILE)
sel_nb11 = pd.read_csv(NB11_SELECTED_FILE)
print('PASS - dataset and frozen splits verified.')
print('NB03 rows:', len(sel_nb03), '| NB04 rows:', len(sel_nb04), '| NB11 rows:', len(sel_nb11))

# Fail loudly if a schema assumption is wrong, rather than silently using a fallback
for col in ['outer_fold', 'model', 'preprocessing']:
    assert col in sel_nb03.columns, f'NB03 file missing column {col}: {list(sel_nb03.columns)}'
for col in ['n_components', 'C', 'epsilon', 'gamma']:
    assert col in sel_nb03.columns, f'NB03 file missing column {col}: {list(sel_nb03.columns)}'
assert {'outer_fold', 'model', 'selected_preprocessing', 'selected_epoch'} <= set(sel_nb04.columns), list(sel_nb04.columns)
assert {'outer_fold', 'preprocessing', 'selected_epoch'} <= set(sel_nb11.columns), list(sel_nb11.columns)
print('PASS - all three selection files have the expected schema.')


In [ ]:
# Data, preprocessing and model builders (identical definitions to the frozen workflow)
df = pd.read_csv(DATA_FILE)
outer = pd.read_csv(OUTER_FILE)

spec_cols = sorted([c for c in df.columns if c.startswith('Spectra_')],
                   key=lambda c: float(c.replace('Spectra_', '')))
X_all = df[spec_cols].to_numpy(dtype=np.float32)
y_all = df['storage_days'].to_numpy(dtype=np.float32)
assert X_all.shape == (660, 331)
N_FEATURES = X_all.shape[1]

class Prep:
    def __init__(self, name):
        self.name, self.reference_, self.scaler_ = name, None, None
    def _base(self, X):
        X = np.asarray(X, dtype=np.float64)
        if self.name == 'raw':  return X.copy()
        if self.name == 'snv':
            mu = X.mean(axis=1, keepdims=True); sd = X.std(axis=1, ddof=1, keepdims=True)
            return (X - mu) / np.where(sd < 1e-12, 1.0, sd)
        if self.name == 'msc':
            ref = self.reference_; rm = ref.mean(); rc = ref - rm; den = np.dot(rc, rc)
            out = np.empty_like(X)
            for i, x in enumerate(X):
                xm = x.mean(); b = np.dot(rc, x - xm) / den
                b = 1.0 if abs(b) < 1e-12 else b
                out[i] = (x - (xm - b*rm)) / b
            return out
        if self.name == 'sg_smooth':
            return savgol_filter(X, SG_WINDOW, SG_POLYORDER, deriv=0, axis=1, mode='interp')
        if self.name == 'sg_deriv1':
            return savgol_filter(X, SG_WINDOW, SG_POLYORDER, deriv=1, delta=1.0, axis=1, mode='interp')
        raise ValueError(self.name)
    def fit(self, X):
        if self.name == 'msc': self.reference_ = np.asarray(X, dtype=np.float64).mean(axis=0)
        self.scaler_ = StandardScaler().fit(self._base(X)); return self
    def transform(self, X):
        return self.scaler_.transform(self._base(X)).astype(np.float32)
    def fit_transform(self, X):
        return self.fit(X).transform(X)

def set_seeds(s=2026):
    random.seed(s); np.random.seed(s); tf.keras.utils.set_random_seed(s)

def build_keras(name):
    if name == 'ANN':
        inp = keras.Input(shape=(N_FEATURES,))
        x = layers.Dense(ANN_HIDDEN[0], activation='relu')(inp); x = layers.Dropout(DROPOUT)(x)
        x = layers.Dense(ANN_HIDDEN[1], activation='relu')(x);  x = layers.Dropout(DROPOUT)(x)
        out = layers.Dense(1)(x)
    elif name == 'CNN1D':
        inp = keras.Input(shape=(N_FEATURES, 1))
        x = layers.Conv1D(16, 7, padding='same')(inp); x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x); x = layers.MaxPooling1D(2)(x)
        x = layers.Conv1D(32, 5, padding='same')(x);  x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x); x = layers.GlobalAveragePooling1D()(x)
        x = layers.Dense(32, activation='relu')(x);   x = layers.Dropout(DROPOUT)(x)
        out = layers.Dense(1)(x)
    else:
        inp = keras.Input(shape=(N_FEATURES, 1))
        if name == 'SimpleRNN': x = layers.SimpleRNN(RECURRENT_UNITS)(inp)
        elif name == 'LSTM':    x = layers.LSTM(RECURRENT_UNITS)(inp)
        elif name == 'BiLSTM':  x = layers.Bidirectional(layers.LSTM(RECURRENT_UNITS))(inp)
        else: raise ValueError(name)
        x = layers.Dropout(DROPOUT)(x)
        x = layers.Dense(DENSE_AFTER_RECURRENT, activation='relu')(x)
        x = layers.Dropout(DROPOUT)(x)
        out = layers.Dense(1)(x)
    m = keras.Model(inp, out, name=name)
    m.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE), loss='mse',
              metrics=[keras.metrics.MeanAbsoluteError(name='mae')])
    return m

test_eggs  = set(outer.loc[outer['outer_fold'] == TIMING_FOLD, 'sample'])
train_eggs = set(df['sample'].unique()) - test_eggs
assert len(train_eggs) == 24 and len(test_eggs) == 6
tr = df['sample'].isin(train_eggs).to_numpy(); te = df['sample'].isin(test_eggs).to_numpy()
print('Timing fold', TIMING_FOLD, '| train rows', tr.sum(), '| test rows', te.sum())


In [ ]:
# Fit each of the seven models once, using the frozen preprocessing/epoch selections
fitted = {}   # name -> dict(model, prep, X_test_ready, complexity_measure, complexity_value)

# ---- PLSR and SVR: refit with the frozen NB03 fold configuration ----
# Column names come from the actual NB03 file (preprocessing, n_components, C, epsilon, gamma).
# No fallback defaults: a wrong epsilon silently changes the support-vector count and therefore
# the measured SVR latency, so the run must fail instead of guessing.
r = sel_nb03[(sel_nb03['outer_fold'] == TIMING_FOLD) & (sel_nb03['model'] == 'PLSR')]
assert len(r) == 1, f'Expected exactly one PLSR row for fold {TIMING_FOLD}, got {len(r)}'
r = r.iloc[0]
plsr_prep_name, plsr_ncomp = str(r['preprocessing']), int(r['n_components'])
pp = Prep(plsr_prep_name); Xtr = pp.fit_transform(X_all[tr]); Xte = pp.transform(X_all[te])
plsr = PLSRegression(n_components=plsr_ncomp).fit(Xtr, y_all[tr])
fitted['PLSR'] = dict(model=plsr, X=Xte, kind='sklearn',
                      measure='Latent components', value=plsr_ncomp)
print(f'PLSR fitted  | prep={plsr_prep_name} | components={plsr_ncomp}')

r = sel_nb03[(sel_nb03['outer_fold'] == TIMING_FOLD) & (sel_nb03['model'] == 'SVR')]
assert len(r) == 1, f'Expected exactly one SVR row for fold {TIMING_FOLD}, got {len(r)}'
r = r.iloc[0]
svr_prep_name = str(r['preprocessing'])
svr_C, svr_eps, svr_gam = float(r['C']), float(r['epsilon']), float(r['gamma'])
pp = Prep(svr_prep_name); Xtr = pp.fit_transform(X_all[tr]); Xte = pp.transform(X_all[te])
svr = SVR(kernel='rbf', C=svr_C, epsilon=svr_eps, gamma=svr_gam).fit(Xtr, y_all[tr])
fitted['SVR'] = dict(model=svr, X=Xte, kind='sklearn',
                     measure='Support vectors', value=int(svr.support_vectors_.shape[0]))
print(f'SVR fitted   | prep={svr_prep_name} | C={svr_C} eps={svr_eps} gamma={svr_gam} '
      f'| support vectors={fitted["SVR"]["value"]}')

# ---- Keras models: frozen preprocessing + frozen epoch count from NB04 / NB11 ----
for name in ['ANN', 'SimpleRNN', 'LSTM', 'BiLSTM', 'CNN1D']:
    if name == 'CNN1D':
        row = sel_nb11[sel_nb11['outer_fold'] == TIMING_FOLD]
        prep_name = str(row.iloc[0]['preprocessing']); epochs = int(row.iloc[0]['selected_epoch'])
    else:
        row = sel_nb04[(sel_nb04['outer_fold'] == TIMING_FOLD) & (sel_nb04['model'] == name)]
        prep_name = str(row.iloc[0]['selected_preprocessing']); epochs = int(row.iloc[0]['selected_epoch'])

    pp = Prep(prep_name); Xtr = pp.fit_transform(X_all[tr]); Xte = pp.transform(X_all[te])
    if name != 'ANN':
        Xtr_m, Xte_m = Xtr[..., np.newaxis], Xte[..., np.newaxis]
    else:
        Xtr_m, Xte_m = Xtr, Xte

    tf.keras.backend.clear_session(); gc.collect(); set_seeds(2026)
    m = build_keras(name)
    t0 = time.perf_counter()
    m.fit(Xtr_m, y_all[tr], epochs=epochs, batch_size=BATCH_SIZE, verbose=0, shuffle=True)
    fitted[name] = dict(model=m, X=Xte_m, kind='keras',
                        measure='Trainable parameters',
                        value=int(sum(np.prod(w.shape) for w in m.trainable_weights)))
    print(f'{name:<10} fitted | prep={prep_name} | epochs={epochs} | params={fitted[name]["value"]:,} | fit={time.perf_counter()-t0:.0f}s')

print('\nAll seven models fitted on CPU.')


In [ ]:
# Serialized model size on disk — measured identically for all seven
import joblib
SIZE_DIR = RESULT_DIR / '_size_probe'
if SIZE_DIR.exists(): shutil.rmtree(SIZE_DIR)
SIZE_DIR.mkdir(parents=True, exist_ok=True)

for name, d in fitted.items():
    if d['kind'] == 'sklearn':
        fp = SIZE_DIR / f'{name}.joblib'; joblib.dump(d['model'], fp)
    else:
        fp = SIZE_DIR / f'{name}.keras';  d['model'].save(fp)
    d['size_mb'] = fp.stat().st_size / (1024**2)
    print(f'{name:<10} {d["size_mb"]:.3f} MB  ({fp.name})')


In [ ]:
# Unified CPU latency, THREE inference paths per model, identical protocol and process
#
#   predict : high-level API            (sklearn .predict / keras .predict)
#   eager   : direct call, no graph     (sklearn .predict / keras model(x, training=False))
#   graph   : direct call, compiled     (sklearn .predict / keras tf.function(model))
#
# The graph column is the one to report. It removes the predict() wrapper overhead AND the
# eager per-op dispatch penalty, so what remains is the model's actual compiled compute cost.
# For scikit-learn all three are the same call, so those rows coincide by construction.

def time_path(d, X, mode, n_warmup, n_repeats):
    """Median single-spectrum latency in ms for one inference path."""
    is_keras = (d['kind'] == 'keras')

    # Pre-build every input outside the timed region so tensor construction is never measured.
    if is_keras and mode in ('eager', 'graph'):
        inputs = [tf.constant(X[i % len(X)][np.newaxis, ...]) for i in range(n_repeats)]
        warm = inputs[0]
    else:
        inputs = [X[i % len(X)][np.newaxis, ...] for i in range(n_repeats)]
        warm = inputs[0]

    if is_keras and mode == 'graph':
        sig = [tf.TensorSpec(shape=(1,) + tuple(X.shape[1:]), dtype=tf.float32)]
        fn = tf.function(lambda t: d['model'](t, training=False), input_signature=sig)
        call = fn
    elif is_keras and mode == 'eager':
        call = lambda t: d['model'](t, training=False)
    elif is_keras:
        call = lambda t: d['model'].predict(t, verbose=0)
    else:
        call = lambda t: d['model'].predict(t)

    for _ in range(n_warmup):          # warm-up: traces the graph once, then reuses it
        call(warm)

    samples = []
    for t in inputs:
        t0 = time.perf_counter()
        call(t)
        samples.append((time.perf_counter() - t0) * 1000.0)
    return np.array(samples)


rows = []
for name in MODELS:
    d = fitted[name]
    X = d['X']

    med = {}
    for mode in ('predict', 'eager', 'graph'):
        s = time_path(d, X, mode, N_TIMING_WARMUP, N_TIMING_REPEATS)
        med[mode] = float(np.median(s))
        if mode == 'graph':
            p25, p75 = float(np.percentile(s, 25)), float(np.percentile(s, 75))

    rows.append({
        'model': name,
        'complexity_measure': d['measure'],
        'value': d['value'],
        'size_MB': round(d['size_mb'], 3),
        'latency_graph_ms': round(med['graph'], 4),
        'latency_eager_ms': round(med['eager'], 4),
        'latency_predict_ms': round(med['predict'], 3),
        'graph_p25_ms': round(p25, 4),
        'graph_p75_ms': round(p75, 4),
        'throughput_graph_spectra_per_s': round(1000.0 / med['graph'], 1),
        'predict_over_graph_ratio': round(med['predict'] / med['graph'], 1),
        'eager_over_graph_ratio': round(med['eager'] / med['graph'], 1),
        'n_timed_repeats': N_TIMING_REPEATS,
    })
    print(f'{name:<10} graph {med["graph"]:9.4f} ms | eager {med["eager"]:9.3f} ms '
          f'| predict {med["predict"]:8.3f} ms | {1000.0/med["graph"]:9.1f} spectra/s')

bench = pd.DataFrame(rows).sort_values('latency_graph_ms').reset_index(drop=True)
bench.to_csv(RESULT_DIR / 'NB15_unified_computational_benchmark.csv', index=False)
print()
display(bench)


In [ ]:
# Emit the replacement Table 8B in manuscript column order, with BOTH latency paths
order = ['SVR', 'PLSR', 'ANN', 'BiLSTM', 'CNN1D', 'LSTM', 'SimpleRNN']
t8b = bench.set_index('model').loc[order].reset_index()
t8b_out = t8b[['model', 'complexity_measure', 'value', 'size_MB',
               'latency_graph_ms', 'throughput_graph_spectra_per_s', 'latency_predict_ms']]
t8b_out.columns = ['Model', 'Complexity measure', 'Value', 'Size (MB)',
                   'CPU latency (ms/spectrum)', 'Throughput (spectra/s)',
                   'High-level API latency (ms/spectrum)']
t8b_out.to_csv(RESULT_DIR / 'NB15_TABLE_8B_replacement.csv', index=False)
print('Replacement Table 8B (paste these values into the manuscript):')
display(t8b_out)

# Parameter-count audit: the manuscript quotes CNN1D as 4,001 "trainable" parameters, but that
# figure is the TOTAL including BatchNormalization moving statistics. Report both explicitly.
print('\nParameter-count audit (trainable vs total):')
audit = []
for name in ['ANN', 'SimpleRNN', 'LSTM', 'BiLSTM', 'CNN1D']:
    m = fitted[name]['model']
    tr = int(sum(np.prod(w.shape) for w in m.trainable_weights))
    nt = int(sum(np.prod(w.shape) for w in m.non_trainable_weights))
    audit.append({'model': name, 'trainable': tr, 'non_trainable': nt, 'total': tr + nt})
audit = pd.DataFrame(audit)
audit.to_csv(RESULT_DIR / 'NB15_parameter_count_audit.csv', index=False)
display(audit)

# SVR support-vector audit across all five folds, to resolve the 288-vs-381 discrepancy
print('\nSVR support-vector count per outer fold (frozen NB03 configuration).')
print('Table 8B currently reports 288; check whether that matches fold 1 or the across-fold mean,')
print('and make the choice explicit in the caption so complexity and latency use the same fold.')
sv_rows = []
for fold in range(1, 6):
    rr = sel_nb03[(sel_nb03['outer_fold'] == fold) & (sel_nb03['model'] == 'SVR')].iloc[0]
    te_f = set(outer.loc[outer['outer_fold'] == fold, 'sample'])
    tr_f = df['sample'].isin(set(df['sample'].unique()) - te_f).to_numpy()
    ppf = Prep(str(rr['preprocessing']))
    Xtr_f = ppf.fit_transform(X_all[tr_f])
    mf = SVR(kernel='rbf', C=float(rr['C']), epsilon=float(rr['epsilon']),
             gamma=float(rr['gamma'])).fit(Xtr_f, y_all[tr_f])
    sv_rows.append({'outer_fold': fold, 'preprocessing': str(rr['preprocessing']),
                    'C': float(rr['C']), 'epsilon': float(rr['epsilon']),
                    'gamma': float(rr['gamma']),
                    'support_vectors': int(mf.support_vectors_.shape[0])})
sv = pd.DataFrame(sv_rows)
sv.to_csv(RESULT_DIR / 'NB15_svr_support_vector_audit.csv', index=False)
display(sv)
print('mean support vectors across folds:', round(sv['support_vectors'].mean(), 1))


In [ ]:
# Regenerate manuscript Figure 6 (accuracy-latency reference plane) from frozen outputs.
#
# Nothing here is hard-coded. The x axis is the compiled-graph latency measured above; the y axis
# is the pooled seed-mean OOF MAE read from the frozen seven-model performance table, so the
# published figure can be rebuilt from the repository without rerunning the modelling notebooks.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PERF_FILE = find_one('Table_R1_performance_7models.csv')
print('MAE source:', PERF_FILE.relative_to(PROJECT_ROOT))
perf = pd.read_csv(PERF_FILE).set_index('model')

# Fixed colours so the figure stays visually consistent across reruns
COLOR = {'PLSR': '#ff7f0e', 'SVR': '#1f77b4', 'ANN': '#2ca02c', 'CNN1D': '#17becf',
         'SimpleRNN': '#8c564b', 'LSTM': '#9467bd', 'BiLSTM': '#d62728'}

# Labels sit to the right of every marker. PLSR and SimpleRNN take a small upward offset because
# their nearest neighbour (SVR and LSTM) is close on the log axis and the text would overlap it.
LABEL_DY = {'PLSR': 7, 'SimpleRNN': 7, 'ANN': 7}
# ANN is pushed up because on some CPU assignments its latency lands very close to SVR's
# (both sub-millisecond); without the offset the two labels overlap.

lat = bench.set_index('model')['latency_graph_ms']
missing = [m for m in COLOR if m not in perf.index or m not in lat.index]
assert not missing, f'Missing series for: {missing}'

fig, ax = plt.subplots(figsize=(6.29, 4.28), dpi=600)
for name in COLOR:
    x, y = float(lat[name]), float(perf.loc[name, 'MAE_days'])
    ax.scatter(x, y, s=70, color=COLOR[name], zorder=3, edgecolors='none')
    ax.annotate(name, (x, y), textcoords='offset points',
                xytext=(12, LABEL_DY.get(name, -4)),
                fontsize=11, color='black', ha='left', va='center')

ax.set_xscale('log')
ax.set_xlabel('CPU inference latency (ms/spectrum, batch size 1; log scale)', fontsize=11)
ax.set_ylabel('Pooled seed-mean OOF MAE (days)', fontsize=11)
ax.set_xlim(0.1, 70)
ax.set_ylim(2.0, 5.25)
ax.grid(True, which='major', color='#e6e6e6', linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
for sp in ax.spines.values():
    sp.set_linewidth(0.9)
    sp.set_color('#333333')
ax.tick_params(labelsize=10)
fig.tight_layout()

FIG_PNG = RESULT_DIR / 'NB15_Figure6_accuracy_latency_plane.png'
FIG_TIF = RESULT_DIR / 'NB15_Figure6_accuracy_latency_plane.tif'
fig.savefig(FIG_PNG, dpi=600, facecolor='white')

# TIFF for submission: RGB with LZW compression, matching the size of the previous figure file
from PIL import Image
Image.open(FIG_PNG).convert('RGB').save(FIG_TIF, compression='tiff_lzw', dpi=(600, 600))
plt.close(fig)

print('Figure 6 written:')
print('  ', FIG_PNG.name, round(FIG_PNG.stat().st_size / 1024**2, 2), 'MB')
print('  ', FIG_TIF.name, round(FIG_TIF.stat().st_size / 1024**2, 2), 'MB')

# Plotted values, for the record
fig6_data = pd.DataFrame({
    'model': list(COLOR),
    'latency_graph_ms': [float(lat[m]) for m in COLOR],
    'pooled_seedmean_MAE_days': [float(perf.loc[m, 'MAE_days']) for m in COLOR],
})
fig6_data.to_csv(RESULT_DIR / 'NB15_Figure6_plotted_values.csv', index=False)
display(fig6_data)


## Reading the result

**Report the graph column in Table 8B.** It is compiled execution without the `predict()`
wrapper, which is how a deployed model actually runs. The other two columns are diagnostic:

- `predict()` minus graph = per-call API overhead. Roughly constant, and large enough that it
  hides real architectural differences when used alone.
- eager minus graph = the cost of running without compilation. Small for feed-forward models,
  severe for recurrent ones because each of the 331 timesteps dispatches separately.

Quoting either diagnostic column as if it were the model's cost is what produced the confusion
in v2 and v3. Keep them in the results package for transparency, keep only the graph column in
the manuscript table, and state in the caption which column is reported.

The manuscript's existing caveat still stands: these are environment-specific reference values
measured on one CPU, not SCiO, smartphone or embedded-device benchmarks.

### Two numbers to settle before editing the manuscript

1. **CNN1D parameters.** The audit prints trainable, non-trainable and total. Section 2.6
   currently says "4,001 trainable parameters"; if total and trainable differ, the wording must
   distinguish them.
2. **SVR support vectors.** The per-fold audit shows how the count varies with each fold's frozen
   epsilon. Decide whether Table 8B reports the fold used for timing or an across-fold summary,
   apply that choice to every row, and say so in the caption.


In [ ]:
# Protocol, summary and result package
protocol = {
    'notebook': 'NB15_UNIFIED_COMPUTATIONAL_BENCHMARK',
    'run_revision': RUN_REVISION,
    'purpose': 'Re-measure Table 8B computational descriptors for all seven models in one consistent CPU environment',
    'dataset_sha256': dataset_sha,
    'frozen_split_manifest_sha256': split_sha,
    'device': 'CPU only (CUDA_VISIBLE_DEVICES=-1)',
    'timing_fold': TIMING_FOLD,
    'timing_warmup_predictions': N_TIMING_WARMUP,
    'timing_repeats': N_TIMING_REPEATS,
    'timing_statistic': 'median of single-spectrum latencies, batch size 1',
    'inference_paths': {
        'predict': 'high-level API (sklearn .predict / keras .predict)',
        'eager': 'direct call without graph (keras model(x, training=False))',
        'graph': 'direct call compiled with tf.function and a fixed input_signature',
    },
    'primary_latency_column': 'graph',
    'input_tensors_prebuilt_outside_timed_region': True,
    'v3_defect_corrected': 'v3 labelled its direct path tf.function-wrapped but used a plain lambda, so it ran eager and penalised recurrent models',
    'models': MODELS,
    'selection_sources': {
        'NB03': str(NB03_SELECTED_FILE.relative_to(PROJECT_ROOT)),
        'NB04': str(NB04_SELECTED_FILE.relative_to(PROJECT_ROOT)),
        'NB11': str(NB11_SELECTED_FILE.relative_to(PROJECT_ROOT)),
    },
    'fold1_PLSR': {'preprocessing': plsr_prep_name, 'n_components': plsr_ncomp},
    'fold1_SVR': {'preprocessing': svr_prep_name, 'C': svr_C, 'epsilon': svr_eps, 'gamma': svr_gam},
    'parameter_count_audit': 'NB15_parameter_count_audit.csv (trainable vs total per model)',
    'svr_support_vector_audit': 'NB15_svr_support_vector_audit.csv (per outer fold)',
    'figure6_outputs': [
        'NB15_Figure6_accuracy_latency_plane.png',
        'NB15_Figure6_accuracy_latency_plane.tif',
        'NB15_Figure6_plotted_values.csv',
    ],
    'figure6_axes_sources': {
        'x': 'latency_graph_ms from this notebook',
        'y': 'MAE_days from Table_R1_performance_7models.csv (frozen)',
    },
    'predictive_results_modified': False,
    'note': 'Predictive metrics, statistical tests and OOF predictions are untouched by this notebook.',
}
(RESULT_DIR / 'NB15_protocol.json').write_text(json.dumps(protocol, indent=2), encoding='utf-8')
(RESULT_DIR / 'NB15_run_summary.json').write_text(json.dumps({
    'status': 'COMPLETED', 'run_revision': RUN_REVISION, 'n_models': len(MODELS),
    'completed_at_utc': datetime.now(timezone.utc).isoformat()}, indent=2), encoding='utf-8')

with open(RESULT_DIR / 'environment_packages.txt', 'w', encoding='utf-8') as f:
    f.write(f'Python: {sys.version}\nPlatform: {platform.platform()}\nTensorFlow: {tf.__version__}\n\n')
    try:
        f.write(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True, stderr=subprocess.STDOUT))
    except Exception as e:
        f.write(f'pip freeze failed: {e}\n')

try:
    cpuinfo = subprocess.check_output(['lscpu'], text=True, stderr=subprocess.STDOUT)
except Exception as e:
    cpuinfo = f'lscpu unavailable: {e}'
(RESULT_DIR / 'hardware_info.txt').write_text(cpuinfo, encoding='utf-8')

if SIZE_DIR.exists(): shutil.rmtree(SIZE_DIR)   # drop the serialized probe files

ZIP_NAME = 'NB15_RESULTS_UNIFIED_COMPUTATIONAL_BENCHMARK.zip'
zip_path = ZIP_DIR / ZIP_NAME
if zip_path.exists(): zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', root_dir=RESULT_DIR)
print('ZIP created:', zip_path, '|', round(zip_path.stat().st_size/1024, 1), 'KB')

from google.colab import files as colab_files
colab_files.download(str(zip_path))
